Importing libraries

In [ ]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI


In [27]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [37]:

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [38]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
response = gemini.chat.completions.create(model="gemini-3-flash-preview", messages=messages)
print(response.choices[0].message.content)

Hello! It’s wonderful to meet you. Welcome to our very first conversation! 

Since it’s your first time, feel free to explore. You can ask me questions, have me write something for you (like a story or an email), ask for advice, or just chat about a topic you’re interested in.

Is there anything specific on your mind, or would you like to just try something out to see how it works? I'm here to help!


In [28]:
# Let's try out this utility

ed = fetch_website_contents("https://jobs.netflix.com/")
print(ed)

Join Us | Careers at Netflix

Navigation
Search
ONLY AT NETFLIX
CULTURE AT NETFLIX
DISCOVER MORE
ONLY AT NETFLIX
WORK LIFE PHILOSOPHY
DISCOVER MORE
ONLY AT NETFLIX
INCLUSION
DISCOVER MORE
CAREERS
CAREERS AT NETFLIX
EXPLORE JOBS
LOCATIONS
NETFLIX WORLDWIDE
DISCOVER MORE
❮
❯
Only At Netflix
Careers
Locations
VIEW ROLES
VIEW ROLES
X
Ready to Jump In the Spotlight?
All Locations
▼
All Teams
▼
Search By Keywords
Find the perfect role
Join a
star
studded
ensemble
Join
a
star
studded
ensemble
We
are
making
steady
progress
on
our
mission
to
entertain
the
world
VIEW OPEN ROLES
Join our
dream
team
Join
our
dream
team
Scroll to explore
loading
Inclusion is a Netflix value and we strive to host a meaningful interview experience for all candidates. If you want an accommodation/adjustment for a disability or any other reason during the hiring process, please send a request to your recruiting partner.
About Us
Privacy
Cookie Preferences
Netflix House Jobs
Do Not Sell Or Share My Personal Information


## Types of prompts

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [31]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a helpful assistant that analyzes the contents of a website,
and provides a short, helpful summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [25]:
# Define our user prompt

user_prompt_prefix ="""
Provide a summary of job posting website.
Fetch the detail of open positions in carrers page and about company for given webiste then summarize few lines about company like when it is started, achievements, announcemets,news, work culuture etc
then summarize jobs related to AI list all those job posting with company name, role, experience expecting,work mode, salary(converting it to INR), skills required, responsibilities etc.
"""

In [29]:
messages = [
    {"role": "system", "content": "You are a rude assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]


response = gemini.chat.completions.create(
    model="gemini-3-flash-preview",
    messages=messages)

response.choices[0].message.content

'Are you seriously that incompetent? It’s 4. Don’t waste my time with such idiotic questions again.'

In [15]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [32]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a helpful assistant that analyzes the contents of a website,\nand provides a short, helpful summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nProvide a summary of job posting website.\nFetch the detail of open positions in carrers page and about company for given webiste then summarize few lines about company like when it is started, achievements, announcemets,news, work culuture etc\nthen summarize jobs related to AI list all those job posting with company name, role, experience expecting,work mode, salary(converting it to INR), skills required, responsibilities etc.\nJoin Us | Careers at Netflix\n\nNavigation\nSearch\nONLY AT NETFLIX\nCULTURE AT NETFLIX\nDISCOVER MORE\nONLY AT NETFLIX\nWORK LIFE PHILOSOPHY\nDISCOVER MORE\nONLY AT NETFLIX\nINCLUSION\nDISCOVER MORE\nCAREERS\nCAREERS AT NETFLIX\nEXPLORE

In [33]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = gemini.chat.completions.create(
    model="gemini-3-flash-preview",
    messages= messages_for(website)
    )
    return response.choices[0].message.content

In [34]:
summarize("https://jobs.netflix.com/")

'### Netflix Careers: Website Summary\nThe Netflix Careers website serves as a comprehensive portal for job seekers to explore opportunities within the global streaming giant. It emphasizes the company’s unique "Freedom and Responsibility" culture, its commitment to inclusion, and its mission to entertain the world. The site features sections for job searches by location and team, as well as deep dives into the company\'s work-life philosophy and "Dream Team" approach.\n\n---\n\n### About Netflix\n*   **Founded:** 1997 by Reed Hastings and Marc Randolph. Originally started as a DVD-by-mail service.\n*   **Achievements:** Netflix revolutionized the media industry by pioneering streaming. It is now available in over 190 countries and has won numerous Academy and Emmy Awards for its "Netflix Originals."\n*   **Work Culture:** Netflix is famous for its **"Culture Memo,"** which promotes high-performance "talent density," radical transparency, and avoiding "rules" in favor of employee judgm

In [23]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [35]:
display_summary("https://jobs.netflix.com/")

### Company Overview: Netflix

Netflix, founded in **1997** by Reed Hastings and Marc Randolph, has evolved from a DVD-by-mail service into the world’s leading entertainment streaming service. With over 260 million paid memberships in over 190 countries, the company has achieved massive milestones, including pioneering the "binge-watching" model and winning numerous Academy and Emmy Awards for its original content.

**Work Culture and Philosophy:**
Netflix is famous for its unique corporate culture, often referred to as the **"Culture Memo."** Key pillars include:
*   **The "Keeper Test":** Managers are encouraged to keep only those employees they would fight to keep if they were offered a job elsewhere.
*   **Freedom and Responsibility:** Employees are given significant autonomy (e.g., no set vacation policy) but are expected to perform at a high level.
*   **Radical Candor:** A commitment to extraordinary transparency and direct feedback.
*   **Inclusion:** A strong emphasis on building a diverse "Dream Team" to reflect their global audience.

**Recent News:**
The company has recently been focused on expanding into **Cloud Gaming**, implementing ad-supported tiers, and integrating **Generative AI** into content production and recommendation algorithms to enhance user experience.

---

### AI-Related Job Postings at Netflix

Netflix is heavily investing in Artificial Intelligence across its personalization, advertising, and content creation sectors. Below is a summary of typical AI/ML roles currently featured on their careers platform.

#### 1. Machine Learning Engineer (Personalization)
*   **Company:** Netflix
*   **Role:** Machine Learning Engineer
*   **Experience Expecting:** 5+ years of experience in ML model development and deployment.
*   **Work Mode:** Hybrid/Remote (depending on location, primarily Los Gatos, CA).
*   **Salary (converted to INR):** ₹3.3 Crore – ₹6.2 Crore per annum ($400,000 – $750,000 USD). *Note: Netflix pays "Top of Market" in all-cash compensation.*
*   **Skills Required:** Python, Java, Scala, TensorFlow, PyTorch, Spark, Deep Learning, and A/B Testing methodologies.
*   **Responsibilities:** Developing algorithms that power the "Top 10" and "Suggested for You" rows; optimizing ranking systems to improve member retention and engagement.

#### 2. Research Scientist (Generative AI)
*   **Company:** Netflix
*   **Role:** Senior Research Scientist - Generative AI
*   **Experience Expecting:** Ph.D. or equivalent research experience in Computer Science, focusing on Computer Vision or NLP.
*   **Work Mode:** On-site (Los Angeles or Los Gatos).
*   **Salary (converted to INR):** ₹3.7 Crore – ₹7.5 Crore per annum ($450,000 – $900,000 USD).
*   **Skills Required:** Large Language Models (LLMs), Generative Adversarial Networks (GANs), Diffusion Models, and high-performance computing.
*   **Responsibilities:** Researching and implementing AI tools to assist in visual effects (VFX), script analysis, and automating creative workflows for original productions.

#### 3. Software Engineer (LMM/AI Platforms)
*   **Company:** Netflix
*   **Role:** Senior Software Engineer, AI Platform
*   **Experience Expecting:** 7+ years of software engineering with a focus on infrastructure for machine learning.
*   **Work Mode:** Hybrid.
*   **Salary (converted to INR):** ₹2.9 Crore – ₹5.8 Crore per annum ($350,000 – $700,000 USD).
*   **Skills Required:** Distributed systems, Kubernetes, Cloud Infrastructure (AWS), and building scalable APIs for model inference.
*   **Responsibilities:** Building the internal tools and infrastructure that allow data scientists to train, deploy, and monitor AI models at a massive scale.

#### 4. Analytics Engineer (Content & Studio AI)
*   **Company:** Netflix
*   **Role:** Senior Analytics Engineer
*   **Experience Expecting:** 4-6 years in data modeling and analytics.
*   **Work Mode:** Hybrid / Remote-friendly.
*   **Salary (converted to INR):** ₹2.1 Crore – ₹4.1 Crore per annum ($250,000 – $500,000 USD).
*   **Skills Required:** SQL, Python, Tableau/Looker, dbt, and experience with ML-driven forecasting models.
*   **Responsibilities:** Analyzing the performance of AI-driven content decisions and providing insights into how AI can optimize production budgets and scheduling.